# 10.1 · Semantic search

**Owner:** Member 4 · **Branch:** `dev-semantic` · **Code:** `src/discovery/rank/embeddings.py` → class `EmbeddingRanker` · **Writes to:** `eval/results/semantic.json`

**What it means:** A small language model (`all-MiniLM-L6-v2`) turns a text into 384 numbers, so that texts with similar meaning get nearby vectors. The score is the cosine between the query's vector and each repository's. FAISS finds the nearest vectors fast.

**Key idea:** It matches meaning, not words: "stock tracker" can find an "inventory management" README that never says "stock". It is expected to win on paraphrased queries and to lose on technology-heavy ones, where the exact word ("nestjs") matters (hypothesis H2).

Full guide with starter code, and the Results sheet: [team plan](https://claude.ai/artifact/Rst8GHxAbeHavg4JS2RZV3) (tab 10.1 Semantic search). Handoff and setup: `docs/handoff-methods.md`.

### Settings to try
| Setting | Values | Why |
|---|---|---|
| text embedded | name + description vs + first 1,000 README characters | MiniLM reads at most 256 word pieces (≈ 200 words): only the start counts. |
| README chunks | start only vs best of several chunks | Split long READMEs, keep the best-matching chunk per repository. |

**Fixed:** sentence-transformers/all-MiniLM-L6-v2, normalize_embeddings=True, faiss.IndexFlatIP (exact search)

### What we agreed (same for all five)
- **One tokenizer:** `tokenize()` from `discovery.preprocess.text`, for documents *and* queries. Never a private one.
- **One interface:** your class extends `SearchMethod` (`src/discovery/rank/base.py`): `fit(corpus)` → self, `search(query, k)` → `[{"doc_id": full_name, "score": float}]`, best first. `check_hits()` checks it.
- **Tune on the 5 development queries only** (`DEV`, replaced by the real ones in Phase 13). The 20 test queries are for the final number only.
- **Seed 42**, paths from `PATHS`, and don't edit the shared files (`rank/base.py`, `rank/devkit.py`, `preprocess/text.py`, `config/config.yaml`) on your branch: ask Member 6.
- Commit this notebook **with its outputs** on `dev-semantic`. No data files in git.

## 1. Setup: load the corpus and the shared tokenizer

**Check:** it prints 8,223 repositories, and `node.js` and `c#` come out of `tokenize()` in one piece.

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from discovery.config import PATHS, SEED, set_seed
from discovery.preprocess.text import doc_text, tokenize      # the ONE shared tokenizer (Phase 6)
from discovery.rank.base import check_hits
from discovery.rank.devkit import DEV, load_corpus, measure, save_results, show_top
from discovery.rank.embeddings import EmbeddingRanker

set_seed()
corpus = load_corpus()                                   # data/processed/corpus.parquet
ids = corpus["full_name"].tolist()                       # doc_id = full_name, "owner/name"
text = doc_text(corpus).tolist()                         # name + description + topics + README
print(f"{len(corpus):,} repositories |", tokenize("Built with Node.js and C#"))
print("development queries:", *DEV, sep="\n  ")

## 2. Build: Semantic search

Embed every repository once and save the vectors, so nobody has to wait for it again. Unit-length vectors make inner product equal to cosine.

**Check:** it runs once on the whole corpus without errors. Write down how long it took: it goes in the sheet.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb_text = (corpus["name"] + ". " + corpus["description"] + ". " + corpus["readme"].str.slice(0, 1000)).tolist()
E = model.encode(emb_text, batch_size=64, show_progress_bar=True, normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(E.shape[1])            # inner product of unit vectors = cosine
index.add(E)
np.save(PATHS["index"] / "embeddings.npy", E)
faiss.write_index(index, str(PATHS["index"] / "faiss.index"))
print(E.shape)                                   # (≈10,000, 384); a few minutes on a laptop CPU
```
</details>

## 3. Search: a free-text query in, a ranked list out

The query is embedded with the same model; FAISS returns the k nearest repositories and their cosine.

**Check:** the top 10 for `DEV[0]` look plausible when you read their names and descriptions.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
def search(query, k=10):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    s, rows = index.search(q, k)
    return [{"doc_id": ids[j], "score": float(v)} for v, j in zip(s[0], rows[0]) if j >= 0]

search(DEV[0])
```
</details>

## 4. Move it into `src/discovery/rank/embeddings.py` as `EmbeddingRanker`

The stub is already there with the right signature. Fill in `fit` and `search` (and `params`), then load it here with `discovery.rank.embeddings.EmbeddingRanker`: autoreload picks up your edits.

**Check:** `check_hits` passes below, and in a terminal `IR_SLOW=1 pytest tests/test_methods.py -k semantic` passes (it was skipped until now).

<details><summary>The search method, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
    def search(self, query, k=10):
        ...  # the function above, with self.model, self.index, self.ids
```
</details>

In [ ]:
method = EmbeddingRanker().fit(corpus)
check_hits(method.search(DEV[0], 10), k=10)
method.params()

## 5. Similarity-based retrieval: does it find paraphrases?

Ask in words a README wouldn't use, then the opposite: a query that only works if the exact technology word matches.

**Check:** The first three should return inventory, POS / e-commerce and school projects even without those words. Note what happens with "nestjs postgresql": that is H2.

<details><summary>The check, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
for q in ["stock tracker for a small warehouse", "app to sell things in a shop", "manage students and classes",
          "nestjs postgresql"]:
    print(q)
    for h in search(q, 5):
        print(f"   {h['score']:.3f}  {h['doc_id']}")
```
</details>

## 6. Read your top 10 on the development queries

Read them. Write down two good results and two bad ones, and why each bad one ranked high (in the markdown cell under the output).

**Check:** you can say in one sentence why each bad result ranked high.

In [ ]:
show_top(method, corpus, DEV)

*Two good results, two bad results, and why:*

- 
- 
- 
- 

## 7. Tune on the 5 development queries only

Compare the text choices in the table on the development queries. Keep the one with the best mean NDCG@10 and write down the encoding time.

Until Phase 13 gives the development labels, compare settings by reading the top 10 (step 6). When `eval/qrels.csv` has the development labels, use `discovery.evaluation.metrics.ndcg_at_k` and keep the setting with the best mean NDCG@10. **Never tune on the 20 test queries.**

**Check:** a small table: each setting tried, what changed in the top 10 (later: mean NDCG@10). Keep the best and note it in the sheet.

## 8. Speed, results file, hand in

`measure()` times the build and the queries; `save_results()` writes `eval/results/semantic.json` in the format the Results sheet imports. The metrics stay empty until Phase 13–14: then Member 6's `evaluate_method(method, split="test")` fills them in and you import the file in the **Results sheet** tab of the team plan, add your observations and download the PDF.

**Check:** `eval/results/semantic.json` exists; this notebook is committed with its outputs on `dev-semantic`; `pytest` passes.

In [ ]:
speed = measure(lambda: EmbeddingRanker().fit(corpus), DEV)
print(speed)
save_results(method, speed)

## Extra (optional): A bigger model

Try `all-mpnet-base-v2` (768 numbers, ~5× slower). Is the gain in NDCG@10 worth the time? Keep MiniLM as the main result.

## What we found

*(fill in at the end: build time, the best setting, two good and two bad results, and anything surprising)*